# 01 · Patient Triage with Breadth-First Search (BFS)

**Goal:** find the first **20 high-risk patients** in a fair, repeatable way, instead of picking patients at random.

Each patient is a node. BFS visits the nodes in order using a first-in-first-out (FIFO) queue, and checks each patient against **six clinical conditions**:

| Rule | Condition |
|---|---|
| 1 | Glucose > 140 mg/dL |
| 2 | Age > 40 |
| 3 | BMI > 30 |
| 4 | Blood pressure > 90 mmHg |
| 5 | Insulin > 200 μU/mL |
| 6 | Diabetes Pedigree Function > 1.0 |

A patient is flagged if **at least one** condition is true.

In [1]:
import pandas as pd
from collections import deque

df = pd.read_csv("../data/diabetes_cleaned.csv")
print(df.shape)

(768, 9)


In [2]:
# Goal test: count how many of the six conditions a patient meets
def count_conditions(row):
    conditions = [
        row['Glucose'] > 140,                    # Rule 1
        row['Age'] > 40,                         # Rule 2
        row['BMI'] > 30,                         # Rule 3
        row['BloodPressure'] > 90,               # Rule 4
        row['Insulin'] > 200,                    # Rule 5
        row['DiabetesPedigreeFunction'] > 1.0,   # Rule 6
    ]
    return sum(conditions)

def get_risk_level(n_met):
    if n_met >= 5: return "High"
    elif n_met == 4: return "Medium"
    else: return "Low"

### BFS traversal (stops after 20 patients)

In [3]:
queue = deque(range(len(df)))
triaged_20 = []

while queue and len(triaged_20) < 20:
    idx = queue.popleft()
    row = df.loc[idx]
    n_met = count_conditions(row)
    if n_met >= 1:
        triaged_20.append((idx, n_met))

print(f"Patient records scanned to find 20: {idx + 1}")
print(f"Triaged patients found: {len(triaged_20)}")

Patient records scanned to find 20: 26
Triaged patients found: 20


### Top 5 among the 20 triaged patients

In [4]:
top5 = sorted(triaged_20, key=lambda x: -x[1])[:5]

rows = []
for order, (idx, n_met) in enumerate(top5, start=1):
    r = df.loc[idx]
    rows.append({"Order": order, "Glucose": r['Glucose'], "Age": r['Age'], "BMI": r['BMI'],
                 "BloodPressure": r['BloodPressure'], "Insulin": r['Insulin'],
                 "DPF": r['DiabetesPedigreeFunction'],
                 "Conditions met": f"{n_met}/6", "Risk level": get_risk_level(n_met)})
pd.DataFrame(rows).set_index("Order")

,Glucose,Age,BMI,BloodPressure,Insulin,DPF,Conditions met,Risk level
Order,,,,,,,,
1,197.0,53.0,30.5,70.0,543.0,0.158,4/6,Medium
2,189.0,59.0,30.1,60.0,846.0,0.398,4/6,Medium
3,143.0,51.0,36.6,94.0,146.0,0.254,4/6,Medium
4,148.0,50.0,33.6,72.0,30.5,0.627,3/6,Low
5,125.0,54.0,32.0,96.0,30.5,0.232,3/6,Low


### What I learned

- BFS always checks patients in the same order, so the result is the same every time.
- But BFS stops at the first 20 it finds. It does **not** guarantee these are the 20 most severe patients in the whole dataset.
- A heuristic search (like best-first or A*) would be needed to find the most urgent cases first.